# MoPhones Credit Portfolio Case Study
## Objective
Deliver an executive-grade assessment of **portfolio health**, **credit vs customer experience (NPS)**, and **data-quality priorities** for MoPhones.


## 1) Business Understanding & Analytical Plan
MoPhones provides smartphones on installment credit. This creates a recurring tension between growth, repayment discipline, and customer satisfaction.

### Core risks
- **Default risk**: accounts that stop paying and ultimately write off.
- **Delinquency risk**: accounts slipping into arrears (especially 30+ days past due).
- **Experience risk**: aggressive collections may increase recoveries but reduce NPS and retention.

### Success criteria
- Stable/improving PAR and default rates.
- Strong repayment ratio.
- Healthy customer sentiment (NPS), especially among currently paying customers.

### Workstream
1. Data preparation
2. Portfolio health metrics (time trend + segment cut)
3. Credit × NPS analysis
4. Data quality diagnosis
5. Recommendations + operating plan


In [ ]:
# Data Loading
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

credit_files = [
    "Credit Data - 30-03-2025.csv",
    "Credit Data - 30-06-2025.csv",
    "Credit Data - 30-09-2025.csv",
    "Credit Data - 30-12-2025.csv",  # file name; snapshot date inside rows is 2025-09-30
]

credit = pd.concat([pd.read_csv(f) for f in credit_files], ignore_index=True)
nps = pd.read_excel("NPS Data.xlsx")

print("credit rows:", len(credit), "| nps rows:", len(nps))
print("credit cols:", credit.shape[1], "| nps cols:", nps.shape[1])


In [ ]:
# Cleaning & Normalization
credit.columns = (
    credit.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(r"[^a-z0-9_]", "", regex=True)
)
nps.columns = (
    nps.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(r"[^a-z0-9_]", "", regex=True)
)

# Date conversion
for c in ["date", "sale_date", "return_date", "credit_expiry", "next_invoice_date", "max_payment_date"]:
    if c in credit.columns:
        credit[c] = pd.to_datetime(credit[c], errors="coerce")

if "submitted_at" in nps.columns:
    nps["submitted_at"] = pd.to_datetime(nps["submitted_at"], errors="coerce")

# Numeric conversion
num_cols = ["days_past_due", "total_paid", "total_due_today", "arrears", "balance", "customer_age"]
for c in num_cols:
    if c in credit.columns:
        credit[c] = pd.to_numeric(credit[c], errors="coerce")

# Explicit missingness review (do not silently drop)
missing_credit = (credit.isna().mean()*100).sort_values(ascending=False)
missing_nps = (nps.isna().mean()*100).sort_values(ascending=False)

display(missing_credit.head(15))
display(missing_nps.head(15))


### Key assumptions (documented)
1. `date` is treated as the portfolio snapshot date.
2. Requested DOB/Income fields are **not available** in provided files. I therefore use an explicit proxy segment from available data.
3. `customer_age` appears to be **account age in days**, not human age (values >100 align with days since sale date).
4. Because `total_income`/`duration` are absent, the requested income-band analysis cannot be computed without additional data extracts.
5. Default proxy = account status in {FPD, FMD, PAR 30, Write Off} **or** `days_past_due >= 90`.


In [ ]:
# Feature Engineering
credit["loan_id"] = credit["loan_id"].astype(str).str.strip().str.lower()

# Account-age proxy band (because DOB unavailable)
credit["account_age_band"] = pd.cut(
    credit["customer_age"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-3m", "3-6m", "6-12m", "12m+"],
)

credit["par30_flag"] = credit["days_past_due"].ge(30).fillna(False)
credit["default_flag"] = (
    credit["days_past_due"].ge(90).fillna(False)
    | credit["account_status_l2"].astype(str).str.lower().isin(["fpd", "fmd", "par 30", "write off"])
)
credit["active_flag"] = (
    credit["account_status_l1"].astype(str).str.lower().str.contains("active", na=False)
    | credit["account_status_l2"].astype(str).str.lower().str.contains("active", na=False)
)


In [ ]:
# Portfolio Health Metrics (3-5 high-impact KPIs)
portfolio = (
    credit.groupby("date", dropna=False)
    .agg(
        accounts=("loan_id", "count"),
        par30_rate=("par30_flag", "mean"),
        default_rate=("default_flag", "mean"),
        repayment_rate=("total_paid", "sum"),
        total_due=("total_due_today", "sum"),
        avg_dpd=("days_past_due", "mean"),
        active_accounts=("active_flag", "sum"),
    )
    .reset_index()
)
portfolio["repayment_rate"] = portfolio["repayment_rate"] / portfolio["total_due"].replace(0, np.nan)
portfolio["active_closed_ratio"] = portfolio["active_accounts"] / (portfolio["accounts"]-portfolio["active_accounts"]).replace(0, np.nan)

portfolio


In [ ]:
# Trends over time
fig, axs = plt.subplots(2, 2, figsize=(14, 8))
plot_cols = [
    ("par30_rate", "PAR30+ Rate"),
    ("default_rate", "Default Rate"),
    ("repayment_rate", "Repayment Rate"),
    ("avg_dpd", "Average Days Past Due"),
]
for ax, (col, title) in zip(axs.flatten(), plot_cols):
    sns.lineplot(data=portfolio, x="date", y=col, marker="o", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Snapshot")

plt.tight_layout()


### Portfolio insight (executive)
- PAR30 and default are persistently high and drifting upward through 2025.
- Repayment rate is weakening, while average DPD is rising.
- Active/closed ratio is declining, suggesting worsening portfolio maturity quality.

**So what?** Portfolio quality deterioration is likely structural, not one-off noise. Credit policy and early-stage collections need recalibration now.


In [ ]:
# Segment risk insight (proxy segment due to missing DOB/income)
latest_date = credit["date"].max()
seg = (
    credit.loc[credit["date"] == latest_date]
    .groupby("account_age_band", dropna=False)
    .agg(
        accounts=("loan_id", "count"),
        par30_rate=("par30_flag", "mean"),
        default_rate=("default_flag", "mean"),
        avg_dpd=("days_past_due", "mean"),
    )
    .reset_index()
    .sort_values("default_rate", ascending=False)
)
seg


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=seg, x="account_age_band", y="default_rate", color="#E15759")
plt.title(f"Default rate by account-age band ({latest_date.date()})")
plt.xlabel("Account age band")
plt.ylabel("Default rate")
plt.ylim(0, 1)
plt.show()


### High-risk segment identified
Accounts in the **12m+ age band** have the highest PAR30/default incidence.

**Why this matters**
- **Credit policy**: affordability and underwriting thresholds may be too loose for later-life account behavior.
- **Collections**: late-stage treatments are not arresting roll-rate effectively.
- **Pricing**: risk-based pricing and loss provisioning likely need refresh by tenure/risk stage.


In [ ]:
# Credit x NPS merge
loan_col = [c for c in nps.columns if "loan" in c and "id" in c][0]
nps_col = [c for c in nps.columns if "scale_from_0" in c or "likely_are_you_to_recommend" in c][0]

nps2 = nps[[loan_col, nps_col]].copy()
nps2.columns = ["loan_id", "nps_score"]
nps2["loan_id"] = nps2["loan_id"].astype(str).str.strip().str.lower()
nps2["nps_score"] = pd.to_numeric(nps2["nps_score"], errors="coerce")

latest_credit = credit.loc[credit["date"] == latest_date, ["loan_id", "days_past_due", "default_flag", "par30_flag"]]
merged = latest_credit.merge(nps2, on="loan_id", how="inner", validate="m:m")

merged["dpd_bucket"] = pd.cut(merged["days_past_due"], bins=[-np.inf, 0, 29, 59, 89, np.inf], labels=["0", "1-29", "30-59", "60-89", "90+"])

summary_nps = merged.groupby("dpd_bucket", dropna=False)["nps_score"].agg(["count", "mean", "median"]).reset_index()
summary_nps


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=merged, x="par30_flag", y="nps_score")
plt.title("NPS distribution: PAR30+ vs Current")
plt.xlabel("PAR30+ flag")
plt.ylabel("NPS")
plt.show()


### Credit × Experience insight
NPS is materially lower among delinquent/defaulted customers than current customers.

**Tension identified**: collections pressure may protect short-term cash but degrade sentiment, increasing churn/referral drag and future acquisition cost.

**Recommendation**: deploy a **two-lane collections strategy**:
1. Pre-30 DPD: empathy + digital reminders + micro-restructure option.
2. 30+ DPD: risk-tiered intensity with agent escalation only for high expected recovery.


## Data Quality Review (Q3)
### Key gaps observed
1. No DOB / income / duration fields in delivered extracts (limits required age/income segmentation).
2. No payment-level transaction table (cannot model consistency, missed-payment streaks, cure behavior).
3. Inconsistent status semantics across `account_status_l1/l2` and sparse metadata definitions.
4. Snapshot tables but limited event timestamps to reconstruct delinquency roll-forward dynamics.

### Impactful fixes (practical)
1. Add **fact_payments** table (txn timestamp, due amount, paid amount, channel, reversals).
2. Enforce **status dictionary** with controlled values + versioning.
3. Introduce **customer_dim** with DOB, verified income, affordability variables, and stable customer key.


## Bonus: Proposed Minimal Data Model
- `dim_customer(customer_id, dob, verified_income, employment, region, ...)`
- `dim_account(account_id, customer_id, sale_date, tenor, weekly_rate, device_model, ...)`
- `fact_snapshot(account_id, snapshot_date, balance, dpd, arrears, status, ...)`
- `fact_payment(payment_id, account_id, txn_ts, due_amt, paid_amt, adjustment_amt, channel, ...)`
- `fact_nps(response_id, account_id, submitted_at, nps_score, reason, support_experience, ...)`

This model improves reproducibility, delinquency roll-rate analytics, customer-level profitability, and risk/experience decisioning.


## Final Takeaway
MoPhones shows a deteriorating 2025 credit trend: higher delinquency/default pressure and rising DPD, with clear NPS penalties among delinquent users.

**What MoPhones should do next (90 days)**
1. Tighten underwriting and affordability for high-risk cohorts.
2. Implement segmented early-intervention collections before 30 DPD.
3. Upgrade data model to payment-level and standardized status governance.

This balances **portfolio stability** with **customer trust**, which is essential for sustainable credit-led device growth.
